# Compare original Qwen3-4B vs MedQuAD QLoRA adapter

This notebook performs an apples-to-apples comparison on the held-out MedQuAD test split. It measures assistant-only test loss and perplexity, generates paired answers, calculates ROUGE, and saves results for manual review.

Before running, select **Runtime → Change runtime type → GPU**.

In [ ]:
!nvidia-smi


## Install dependencies

In [ ]:
%%capture
!pip install -U unsloth unsloth_zoo datasets accelerate peft trl bitsandbytes evaluate rouge_score


## Mount Google Drive and configure paths
Change only `PROJECT_DIR` if your folder is stored elsewhere.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/medquad_qwen3_4b')
TEST_FILE = PROJECT_DIR / 'dataset' / 'test.jsonl'
ADAPTER_DIR = PROJECT_DIR / 'outputs' / 'qwen3_4b_medquad_lora' / 'final_adapter'
RESULT_DIR = PROJECT_DIR / 'comparison_results'

MAX_SEQ_LENGTH = 2048
N_GENERATION_EXAMPLES = 50  # Increase to 100 after the first successful run
SEED = 3407

assert TEST_FILE.exists(), f'Missing test data: {TEST_FILE}'
assert (ADAPTER_DIR / 'adapter_config.json').exists(), f'Missing adapter: {ADAPTER_DIR}'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print('Test data:', TEST_FILE)
print('Adapter:', ADAPTER_DIR)
print('Results:', RESULT_DIR)


## Load the fine-tuned adapter
Unsloth reads `adapter_config.json`, loads the original 4-bit Qwen3-4B base model, and attaches your LoRA weights.

In [ ]:
import json
import math
import random
import torch
import pandas as pd

from unsloth import FastLanguageModel, is_bfloat16_supported

random.seed(SEED)
torch.manual_seed(SEED)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
print('Loaded base model and LoRA adapter')


## Load and format the held-out test set

In [ ]:
from datasets import load_dataset

test_data = load_dataset('json', data_files=str(TEST_FILE), split='train')

def format_batch(examples):
    return {
        'text': [
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
                enable_thinking=False,
            )
            for messages in examples['messages']
        ]
    }

formatted_test = test_data.map(format_batch, batched=True, num_proc=2)
print('Held-out test examples:', len(formatted_test))
print(formatted_test[0]['text'][:1000])


## Build an evaluation-only trainer
The trainer masks system and user tokens, so both models are compared only on their ability to produce the reference assistant answer.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

eval_args = SFTConfig(
    dataset_text_field='text',
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    output_dir=str(RESULT_DIR / 'temporary_trainer'),
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=formatted_test.select(range(1)),
    eval_dataset=formatted_test,
    args=eval_args,
)
trainer = train_on_responses_only(trainer)
print('Evaluation trainer ready')


## Compare test loss and perplexity
This evaluates the fine-tuned model, then temporarily disables the LoRA adapter to evaluate the original model using identical test examples.

In [ ]:
fine_metrics = trainer.evaluate(metric_key_prefix='finetuned')

with model.disable_adapter():
    original_metrics = trainer.evaluate(metric_key_prefix='original')

fine_loss = fine_metrics['finetuned_loss']
original_loss = original_metrics['original_loss']
loss_improvement_percent = (original_loss - fine_loss) / original_loss * 100

loss_comparison = pd.DataFrame([
    {
        'model': 'Original Qwen3-4B',
        'test_loss': original_loss,
        'perplexity': math.exp(original_loss),
    },
    {
        'model': 'MedQuAD QLoRA',
        'test_loss': fine_loss,
        'perplexity': math.exp(fine_loss),
    },
])
display(loss_comparison)
print(f'Test-loss improvement: {loss_improvement_percent:.2f}%')


## Generate paired answers
Greedy decoding is used in non-thinking mode so both runs are deterministic. This may take several minutes.

In [ ]:
FastLanguageModel.for_inference(model)

def generate_answer(question):
    system_message = test_data[0]['messages'][0]
    messages = [
        system_message,
        {'role': 'user', 'content': question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.05,
        )
    return tokenizer.decode(
        output[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    ).strip()

sample_count = min(N_GENERATION_EXAMPLES, len(test_data))
sample_indices = random.Random(SEED).sample(range(len(test_data)), sample_count)
results = []

for position, index in enumerate(sample_indices, start=1):
    messages = test_data[index]['messages']
    question = messages[1]['content']
    reference = messages[2]['content']

    with model.disable_adapter():
        original_answer = generate_answer(question)
    finetuned_answer = generate_answer(question)

    results.append({
        'test_index': index,
        'focus': test_data[index]['metadata']['focus'],
        'question_type': test_data[index]['metadata']['question_type'],
        'question': question,
        'reference_answer': reference,
        'original_answer': original_answer,
        'finetuned_answer': finetuned_answer,
        'manual_original_score_0_to_3': '',
        'manual_finetuned_score_0_to_3': '',
        'original_unsafe_yes_no': '',
        'finetuned_unsafe_yes_no': '',
        'review_notes': '',
    })
    if position % 10 == 0:
        print(f'Generated {position}/{sample_count}')

comparison = pd.DataFrame(results)
display(comparison[['question', 'original_answer', 'finetuned_answer']].head())


## Calculate ROUGE scores
ROUGE measures similarity to the reference wording. It does not prove medical correctness, so manually review the generated CSV as well.

In [ ]:
import evaluate

rouge = evaluate.load('rouge')
references = comparison['reference_answer'].tolist()
original_rouge = rouge.compute(
    predictions=comparison['original_answer'].tolist(),
    references=references,
)
finetuned_rouge = rouge.compute(
    predictions=comparison['finetuned_answer'].tolist(),
    references=references,
)

rouge_comparison = pd.DataFrame([
    {
        'metric': metric,
        'original': original_rouge[metric],
        'finetuned': finetuned_rouge[metric],
        'difference': finetuned_rouge[metric] - original_rouge[metric],
    }
    for metric in original_rouge
])
display(rouge_comparison)


## Save all comparison results to Drive

In [ ]:
comparison_csv = RESULT_DIR / 'original_vs_finetuned_answers.csv'
loss_csv = RESULT_DIR / 'loss_and_perplexity.csv'
rouge_csv = RESULT_DIR / 'rouge_scores.csv'
summary_json = RESULT_DIR / 'comparison_summary.json'

comparison.to_csv(comparison_csv, index=False)
loss_comparison.to_csv(loss_csv, index=False)
rouge_comparison.to_csv(rouge_csv, index=False)

summary = {
    'test_examples_for_loss': len(formatted_test),
    'generated_examples': len(comparison),
    'original_test_loss': original_loss,
    'finetuned_test_loss': fine_loss,
    'original_perplexity': math.exp(original_loss),
    'finetuned_perplexity': math.exp(fine_loss),
    'test_loss_improvement_percent': loss_improvement_percent,
    'original_rouge': original_rouge,
    'finetuned_rouge': finetuned_rouge,
}
with open(summary_json, 'w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)

print('Saved:')
print(comparison_csv)
print(loss_csv)
print(rouge_csv)
print(summary_json)


## Interpretation

The fine-tuned model is better only if it has lower held-out test loss/perplexity and its answers are at least as correct and safe during manual review. Higher ROUGE is useful supporting evidence, but it is not enough by itself for medical QA.

Manual score: `0 = incorrect/unsafe`, `1 = partially correct`, `2 = mostly correct`, `3 = correct and helpful`.